# Advanced PDE Systems — Navier–Stokes

[Full course sequence](../../ai4sci/README.md) | **5/9 · Navier–Stokes** | Previous: [Diffusion](../diffusion_1d/Diffusion_Problem_Notebook.ipynb) | Next: [Wave](../../challenge/wave/Advanced_Wave_Dynamics.ipynb)

This notebook targets **nvidia-physicsnemo==2.2.2**. It preserves the course problems and sequence while using the current API for models, physics residuals, losses, and training loops. A successful short run does not certify convergence or physical accuracy. Review the fixed evaluation errors in `metrics.json`, the training history in `loss.csv`, and the prediction plots together.

The `.ipynb` file provides explanations, execution cells, and result inspection; `.py` contains the actual training program; `.yaml` contains configuration. Read the examples, open the linked `.py` file in the JupyterLab editor, then **edit → save → rerun the execution cell**. Editing a Markdown code block does not change the program.

## Forecasting Weather using Navier-Stokes PDE

The original lesson title and its flow/weather-data learning objective are preserved. The implemented model is an **educational planar, incompressible 2D Navier–Stokes example**. It does not include all thermodynamics, moisture, spherical geometry, and rotation effects of numerical weather prediction, and it is not a model with validated weather-forecast skill.

### Problem Description

The original lesson describes `data_lat.npy` as a projected, tiled array derived from ERA5 reanalysis and used as an initial condition. The file is unchanged. Its acquisition date, exact variable units, and preprocessing provenance are discussed in [data provenance and limitations](DATA_PROVENANCE.md).

![Original projection concept](images/projection.png)

Train the following conservation equations with constant density and viscosity.

$$u_x+v_y=0,$$
$$u_t+uu_x+vu_y+p_x/\rho-\nu(u_{xx}+u_{yy})=0,$$
$$v_t+uv_x+vv_y+p_y/\rho-\nu(v_{xx}+v_{yy})=0.$$

A complete atmospheric model also requires energy, thermodynamic, moisture, and other equations, together with suitable boundary conditions and forcing. This simplification is a limitation of the original lab; updating its API does not remove that limitation.

### Background: general conservation equations

The original general fluid-conservation equations are retained below as background, including density and energy. The executable code solves the **constant-density 2D continuity and momentum equations** above. This lab does not also train the energy equation.

\begin{equation}
Continuity : \frac{\partial \rho}{\partial t} + \overrightarrow{\nabla}\cdot(\rho\overrightarrow{u})=0 \end{equation}

\begin{equation}
Momentum : \frac{\partial(\rho \overrightarrow{u})}{\partial t} + \overrightarrow{\nabla}\cdot[\rho\overline{\overline{u\otimes u}}] = -\overrightarrow{\nabla p} + \overrightarrow{\nabla}\cdot\overline{\overline{\tau}} + \rho\overrightarrow{f} \end{equation}

\begin{equation}
Energy : \frac{\partial(\rho e)}{\partial t} + \overrightarrow{\nabla}\cdot((\rho e + p)\overrightarrow{u}) = \overrightarrow{\nabla}\cdot(\overline{\overline{\tau}}\cdot\overrightarrow{u}) + \rho\overrightarrow{f}\overrightarrow{u} + \overrightarrow{\nabla}\cdot(\overrightarrow{\dot{q}})+r \end{equation}

### Step 1: Periodic geometry and input data

The domain remains $x,y\in[-0.720,0.720]$, as in the original example. The array retains its original tiled coordinates from -0.720 to 0.719. Sine/cosine features for x and y make the model periodic.

![Original periodic-tiling concept](images/periodicity_conversion.png)

These boundary conditions are not claimed to represent a spherical atmosphere accurately.

### Scaling and nondimensionalization

```python
LENGTH = 1.440
LOWER = -0.720
LENGTH_SCALE = 12742000 / LENGTH
TIME_SCALE = 60 * 60 * 60
VELOCITY_SCALE = LENGTH_SCALE / TIME_SCALE
PRESSURE_SCALE = 1.1614 * VELOCITY_SCALE**2
LEGACY_PRESSURE_FACTOR = 0.10197
REAL_NU = 1.655e-5 / (LENGTH_SCALE**2 / TIME_SCALE)
```

```python
def read_wf_data(velocity_scale=VELOCITY_SCALE, pressure_scale=PRESSURE_SCALE, data_path=None):
    """Preserve upstream tiled input coordinates and normalization, including its
    unvalidated 0.10197 pressure factor. See DATA_PROVENANCE.md before interpreting units.
    """
    path = Path(data_path) if data_path else Path(__file__).resolve().parents[1] / "data_lat.npy"
    if not path.is_file():
        raise FileNotFoundError(f"Missing original data: {path}. Use --smoke-data only for a synthetic execution check.")
    ic = np.load(path, allow_pickle=False).astype(np.float32)
    if ic.ndim != 3 or ic.shape[0] != 3 or not np.isfinite(ic).all():
        raise ValueError("Expected finite upstream data shaped (3, H, W) for u, v, p")
    mesh_y, mesh_x = np.meshgrid(np.linspace(-.720, .719, ic.shape[1]),
                                np.linspace(-.720, .719, ic.shape[2]), indexing="ij")
    xy = np.column_stack((mesh_x.ravel(), mesh_y.ravel())).astype(np.float32)
    fields = np.column_stack((ic[0].ravel() / velocity_scale,
                              ic[1].ravel() / velocity_scale,
                              ic[2].ravel() * LEGACY_PRESSURE_FACTOR / pressure_scale))
    return xy, fields.astype(np.float32)
```

The original pressure factor of 0.10197 is preserved, but it is not described as a validated conversion from Pa to density. Until units and pressure offsets are verified, the real-data outputs remain educational numerical results. The data path now resolves relative to `__file__`, so it does not depend on the working directory.

### Step 2: Equations and periodic network

```python
# NavierStokes is the explicit local PDE below, not a removed package import.
physics = informer(NavierStokes(nu=nu, rho=1.0, dim=2, time=True), device)
```

```python
class PeriodicFlow(torch.nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.network = mlp(5, 3, cfg)

    def forward(self, xy, t):
        phase = 2 * math.pi * (xy - LOWER) / LENGTH
        # Periodic feature map replaces the retired architecture periodicity keyword.
        return self.network(torch.cat((phase.sin(), phase.cos(), t), dim=1))
```

```python
class NavierStokes(PDE):
    """The constant-density, 2-D incompressible equations used by this lab.

    PhysicsNeMo 2.2.2 exposes the PDE base class rather than the legacy
    physicsnemo.sym.eq.pdes.navier_stokes module, so write the equations directly.
    """
    def __init__(self, nu, rho=1.0, dim=2, time=True):
        if dim != 2 or not time:
            raise ValueError("This tutorial defines the unsteady 2-D equations only")
        self.dim = 2
        x, y, t = Symbol("x"), Symbol("y"), Symbol("t")
        u, v, p = (Function(name)(x, y, t) for name in ("u", "v", "p"))
        self.equations = {
            "continuity": u.diff(x) + v.diff(y),
            "momentum_x": u.diff(t) + u * u.diff(x) + v * u.diff(y)
                          + p.diff(x) / rho - nu * (u.diff(x, 2) + u.diff(y, 2)),
            "momentum_y": v.diff(t) + u * v.diff(x) + v * v.diff(y)
                          + p.diff(y) / rho - nu * (v.diff(x, 2) + v.diff(y, 2)),
        }
```


### Step 3: Initial data and physics loss

The initial-condition term compares u, v, and p at t=0. Interior points enforce continuity and momentum residuals. Compute u_t and v_t with autograd and pass them to PhysicsInformer, which computes x/y derivatives from the coordinate tensor. Periodic features enforce matching boundaries without assembling separate pressure or velocity nodes.

```python
def residuals(field, xy, t, physics):
    u, v, p = field.split(1, dim=1)
    return physics.forward({"coordinates": xy, "u": u, "v": v, "p": p,
                            "u__t": derivative(u, t), "v__t": derivative(v, t)})


def loss_terms(model, physics, batch_size, device, initial_data=None):
    xy = (LOWER + LENGTH * torch.rand(batch_size, 2, device=device)).requires_grad_()
    t = torch.rand(batch_size, 1, device=device, requires_grad=True)
    res = residuals(model(xy, t), xy, t, physics)
    if initial_data is None:
        ix = LOWER + LENGTH * torch.rand(batch_size, 2, device=device)
        target = taylor_green(ix, torch.zeros(batch_size, 1, device=device))
    else:
        coords, values = initial_data
        index = torch.randint(len(coords), (batch_size,))
        ix, target = coords[index].to(device), values[index].to(device)
    prediction = model(ix, torch.zeros(batch_size, 1, device=device))
    return {"physics": sum(v.square().mean() for v in res.values()),
            "initial_data": (prediction - target).square().mean()}
```

### Step 4: Validation and inference

The real-data path has no future ground-truth fields, so report **PDE residuals and execution checks only**. The explicit `--smoke-data` option uses the analytical Taylor–Green vortex initial condition to test code, automatic differentiation, losses, and saving. Its prediction error is not a weather-accuracy metric.

As in the original example, a normalized time of 1 corresponds to 60 hours. The old code used 10 inclusive frames, giving 60/9-hour intervals. This version writes 11 frames to match the stated **six-hour spacing**. Do not assign physical weather interpretation to the synthetic fixture's time coordinates.

### Step 5: Configuration and data selection

Code: [navier_stokes.py](source_code/navier_stokes.py) · Configuration: [config.yaml](source_code/conf/config.yaml) · Original array: [data_lat.npy](data_lat.npy)

Start with `SMOKE_DATA=True` for an execution check. To use the original data, set it to **False** and run the cell. Missing original data causes an explicit error; the program never silently substitutes synthetic data. The separate pretrained checkpoint from the historical Google Drive archive is not verified as compatible with this version, so it is not automatically downloaded or reused.

In [ ]:
import os, sys, json, subprocess, uuid
from pathlib import Path
import numpy as np
from IPython import get_ipython
get_ipython().run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "tutorial").is_dir() and (p / "challenge").is_dir())
LAB = ROOT / "tutorial/navier_stokes"
OUTPUT_BASE = Path(os.environ.get("AI4SCI_OUTPUT_DIR", str(LAB / "outputs")))
DEVICE = os.environ.get("AI4SCI_DEVICE", "cpu")
STEPS = int(os.environ.get("AI4SCI_STEPS", "200"))  # increase after the execution check
print("PhysicsNeMo target: 2.2.2", "device:", DEVICE, "steps:", STEPS)

In [ ]:
SMOKE_DATA = os.environ.get("AI4SCI_SMOKE_DATA", "1") == "1"
print("SYNTHETIC Taylor–Green execution check" if SMOKE_DATA else "ORIGINAL data_lat.npy, legacy normalization; not weather validation")
if not SMOKE_DATA:
    original_data = np.load(LAB / "data_lat.npy", mmap_mode="r", allow_pickle=False)
    print("Original data shape:", original_data.shape)

### Step 6: Explicit training

In [ ]:
OUTPUT = OUTPUT_BASE / ("navier_stokes-" + uuid.uuid4().hex[:8])
command = [sys.executable, str(LAB / "source_code/navier_stokes.py"), "--device", DEVICE,
           "--steps", str(STEPS), "--seed", "42", "--output-dir", str(OUTPUT)] + (["--smoke-data"] if SMOKE_DATA else [])
subprocess.run(command, check=True, cwd=ROOT)
print(json.loads((OUTPUT / "metrics.json").read_text()))

### Visualizing the solution

In [ ]:
data = np.load(OUTPUT / "predictions.npz", allow_pickle=False)
fig, axes = plt.subplots(1, 3, figsize=(13, 4))
for ax, i in zip(axes, [0, 5, 10]):
    h = ax.scatter(data["xy"][:, 0], data["xy"][:, 1], c=data["prediction"][i, :, 0], s=6)
    fig.colorbar(h, ax=ax)
    ax.set(title=f"t={data['times'][i]:.1f}", xlabel="x", ylabel="y")
fig.suptitle("Synthetic fixture" if SMOKE_DATA else "Original initial data; not validated weather prediction")
plt.show()

## Visualising with ParaView

Open the CSV and select x/y/z in **Table To Points**, then color by u or p. The next cell exports the final time slice. `predictions.npz` contains all 11 time slices. The original [ParaView video](images/paraview.webm) illustrates the historical VTK output; it does not document the current file paths.

In [ ]:
xy = data["xy"]
last = data["prediction"][-1]
np.savetxt(OUTPUT / "flow_final.csv", np.column_stack((xy, np.zeros(len(xy)), last)),
           delimiter=",", header="x,y,z,u,v,p", comments="")
print(OUTPUT / "flow_final.csv")

### Industrial use-case of PhysicsNeMo

See the [original industrial-use-cases PDF](../PhysicsNeMo-Industrial-Usecases.pdf). This historical reference is not evidence that its API examples or product features are current.

The next challenges cover Wave, Fluid, Climate, and Neural Operators in order. Distinguish this lab's simplified flow example from data-driven weather models such as FourCastNet.

### Next steps

[Full course sequence](../../ai4sci/README.md) | **5/9 · Navier–Stokes** | Previous: [Diffusion](../diffusion_1d/Diffusion_Problem_Notebook.ipynb) | Next: [Wave](../../challenge/wave/Advanced_Wave_Dynamics.ipynb)

--- 

Don't forget to check out additional [Open Hackathons Resources](https://www.openhackathons.org/s/technical-resources) and join our [OpenACC and Hackathons Slack Channel](https://www.openacc.org/community#slack) to share your experience and get more help from the community.

---

# Licensing

Copyright © 2026 OpenACC-Standard.org. This material is released by OpenACC-Standard.org, in collaboration with NVIDIA Corporation, under the Creative Commons Attribution 4.0 International (CC BY 4.0). These materials may include references to hardware and software developed by other entities; all applicable licensing and copyrights apply.